# Text Preprocessing


Notebook ini menjalankan **text preprocessing** pada korpus **60 dokumen** berita *Liga Indonesia* dari **3 media**: IDN Times (20), ANTARA (20), dan Kompas (20).

`Raw Text → Case Folding → Pembersihan Noise (per media) → Tokenization → Filtering → Stopword Removal → Stemming → Clean Tokens`


## Bagian 0 — Persiapan Environment

In [1]:
# Jalankan sekali saja. Jika PySastrawi sudah ter-install, sel ini tidak mengubah apa pun.
import sys
try:
    import Sastrawi  # noqa: F401
except ImportError:
    !{sys.executable} -m pip install -q PySastrawi
    import Sastrawi  # noqa: F401
print("PySastrawi siap digunakan.")

PySastrawi siap digunakan.


In [2]:
import re
import os
import pandas as pd
from collections import Counter
from functools import lru_cache

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

pd.set_option("display.max_colwidth", 120)
print("Library siap.")

Library siap.


## Bagian 1 — Konfigurasi Korpus

- `SHEET_NAME` = sheet yang berisi korpus (`Corpus_Gabungan`). Sheet lain (`Ringkasan`, `Pemeriksaan_Data`) hanya rekap.
- `PERTAHANKAN_NEGASI` = kata negasi (`tidak`, `bukan`, dst.) tidak ikut dibuang saat stopword removal.
- `STOPWORD_TAMBAHAN` = nama media, nama hari, dan nama bulan yang muncul sebagai boilerplate berita, bukan isi berita.

In [4]:
FILE_CORPUS = "Corpus_Liga_Indonesia.xlsx"
SHEET_NAME = "Corpus_Gabungan"

PERTAHANKAN_NEGASI = True
KATA_NEGASI = {"tidak", "bukan", "jangan", "tanpa", "belum"}

PAKAI_STOPWORD_TAMBAHAN = True
STOPWORD_TAMBAHAN = {
    # nama media (muncul di kutipan sumber, mis. "dilansir Kompas.com")
    "kompas", "com", "antara", "idn", "times",
    # nama hari & bulan (muncul di keterangan waktu, mis. "Selasa (11/8/2026)")
    "senin", "selasa", "rabu", "kamis", "jumat", "sabtu", "minggu",
    "januari", "februari", "maret", "april", "mei", "juni", "juli",
    "agustus", "september", "oktober", "november", "desember",
}

## Bagian 2 — Memuat Korpus

Sel ini otomatis meminta upload file kalau `FILE_CORPUS` belum ada di Colab.

In [5]:
if not os.path.exists(FILE_CORPUS):
    try:
        from google.colab import files
        print("Pilih file korpus (.xlsx):")
        uploaded = files.upload()
        FILE_CORPUS = next(iter(uploaded))      # pakai nama file yang barusan di-upload
    except ImportError:
        raise FileNotFoundError(f"File '{FILE_CORPUS}' tidak ditemukan di folder kerja.")

df = pd.read_excel(FILE_CORPUS, sheet_name=SHEET_NAME)
df = df.dropna(subset=["text"]).reset_index(drop=True)
df["date"] = pd.to_datetime(df["date"]).dt.date      # kolom date bercampur string & datetime -> seragamkan

print(f"Jumlah dokumen dimuat dari '{FILE_CORPUS}', sheet '{SHEET_NAME}': {len(df)}")
display(df.groupby("media").agg(jumlah_dokumen=("doc_id", "count"),
                                rata2_karakter=("text", lambda s: int(s.str.len().mean()))))
df[["doc_id", "title", "media", "date"]].head(10)

Pilih file korpus (.xlsx):


Saving Corpus_Liga Indonesia.xlsx to Corpus_Liga Indonesia.xlsx
Jumlah dokumen dimuat dari 'Corpus_Liga Indonesia.xlsx', sheet 'Corpus_Gabungan': 60


,jumlah_dokumen,rata2_karakter
media,,
ANTARA,20,2642
IDN Times,20,2697
Kompas,20,3545


,doc_id,title,media,date
0,IDN001,"Dari Boca Juniors ke Kalteng, Adriel Fernandez Bomber Baru Isenmulang",IDN Times,2026-08-19
1,IDN002,Pesona Gelandang Gambia di Asia Bikin Isenmulang Kalteng FC Kepincut,IDN Times,2026-08-19
2,IDN003,Persija dan Se'Indonesia Perpanjang Kontrak Hingga 2 Musim,IDN Times,2026-08-15
3,IDN004,"I.League: Persija Minta Jamu Persib Lebih Dulu, Main di Jakarta",IDN Times,2026-08-12
4,IDN005,Jadwal Super League Musim 2026/27 Adopsi Kompetisi Top Dunia,IDN Times,2026-08-12
5,IDN006,Cari Talenta Baru Lewat Batam Prime International GFF 2026,IDN Times,2026-08-08
6,IDN007,Malut United Resmi Berganti Nama Jadi Java United FC,IDN Times,2026-08-03
7,IDN008,"Semifinal Piala Presiden Panas, Derby Jatim Hingga Persib vs Persija",IDN Times,2026-08-02
8,IDN009,Semen Padang Minta Semua Tim Championship Tak Curang,IDN Times,2026-07-19
9,IDN010,Ini Kesan Shin Tae Yong Usai Pimpin Latihan Perdana Persija,IDN Times,2026-07-16


## Bagian 3 — Melihat Raw Text per Media

Sebelum diproses, lihat dulu **awal dan akhir** satu dokumen dari tiap media. Tiap media punya "sampah" yang berbeda:
- **IDN Times**: dateline `Kota, IDN Times -`, nama file gambar (`.jpeg`), kredit `(Dok. ...)`, dan `Iklan - Scroll untuk Melanjutkan`.
- **ANTARA**: keterangan foto sebelum dateline `Jakarta (ANTARA) -`, baris `Baca juga: ...`, dan footer `Pewarta / Editor / Copyright`.
- **Kompas**: `KOMPAS.com -`, keterangan `Lihat Foto ... (kredit)`, dan widget `Baca juga: ... Artikel Kompas.id`.

In [6]:
for media in ["IDN Times", "ANTARA", "Kompas"]:
    row = df[df["media"] == media].iloc[0]
    print(f"=== {media} | {row['doc_id']} | {row['title']}")
    print("AWAL :", repr(row["text"][:250]))
    print("AKHIR:", repr(row["text"][-250:]))
    print()

=== IDN Times | IDN001 | Dari Boca Juniors ke Kalteng, Adriel Fernandez Bomber Baru Isenmulang
AWAL : 'Balikpapan, IDN Times - Tim Adhyaksa FC yang sudah bertransformasi jadi Isenmulang Kalteng FC kembali menambah amunisi asing untuk menghadapi kompetisi Super League 2026/2027. Klub asal Kalimantan Tengah itu resmi mendatangkan pesepak bola asal Boliv'
AKHIR: 'hadapi ketatnya persaingan Super League 2026/2027.\nDengan kedatangan Fernandez, Isenmulang Kalteng FC terus memperkuat komposisi skuad sekaligus mempersiapkan diri menghadapi musim baru dengan target bersaing di level tertinggi sepak bola Indonesia\n\n'

=== ANTARA | ANT001 | Championship 2026/2027 bergulir 11 September, dibuka PSIS Lawan PSPS

AWAL : 'Advisor I.League Budiman Dalimunte memberikan keterangan kepada awak media di Jakarta, Senin (24/8/2026). ANTARA/Aloysius Lewokeda\nJakarta (ANTARA) - Operator liga PT Liga Utama Indonesia (LIU) memastikan kompetisi Championship akan bergulir pada 11 '
AKHIR: 'rup A dan Grup B men

## Tahap 1 — Case Folding

Menyeragamkan huruf besar/kecil supaya "Liga", "LIGA", dan "liga" dianggap sama.

In [7]:
def case_folding(text: str) -> str:
    return text.lower()

df["step1_case_folded"] = df["text"].apply(case_folding)

print("SEBELUM:", df.loc[0, "text"][:200])
print()
print("SESUDAH:", df.loc[0, "step1_case_folded"][:200])

SEBELUM: Balikpapan, IDN Times - Tim Adhyaksa FC yang sudah bertransformasi jadi Isenmulang Kalteng FC kembali menambah amunisi asing untuk menghadapi kompetisi Super League 2026/2027. Klub asal Kalimantan Ten

SESUDAH: balikpapan, idn times - tim adhyaksa fc yang sudah bertransformasi jadi isenmulang kalteng fc kembali menambah amunisi asing untuk menghadapi kompetisi super league 2026/2027. klub asal kalimantan ten


## Tahap 1b — Pembersihan Noise Khas Tiap Media

Karena korpus ini gabungan 3 media, aturan pembersihannya **dibuat per media** lalu dipilih lewat kolom `media`. Ada juga aturan umum yang berlaku untuk semuanya.

Catatan desain: widget "Baca juga" Kompas dihapus **sebelum** case folding, karena judulnya ditulis Title Case dan menyatu dengan paragraf; huruf kapital adalah satu-satunya petunjuk di mana judul itu berakhir. Whitespace baru dirapikan di akhir (di ANTARA, pemisah baris `\n` membantu menentukan akhir judul).

In [8]:
def clean_umum(t: str) -> str:
    """Noise yang bisa muncul di media mana pun."""
    t = t.replace("\xa0", " ")
    # nama file gambar yang ikut tercopy (mis. "IMG-2026.jpg", "whatsapp image 2025-09-11 at 20.42.51 (3).jpeg")
    t = re.sub(r"(?:whatsapp image\s+\S+\s+at\s+[\d.]+(?:\s*[ap]m)?(?:\s*\(\d+\))?|\S+)\.(?:jpe?g|png)", " ", t)
    # kredit foto di dalam kurung: (dok. x), (idn times/x), (instagram.com/x), (kompas.com/x), (xinhua/x)
    t = re.sub(r"\((?:dok\.?|idn times\s*/|instagram\.com|persib\.co\.id|kompas\.com\s*/|xinhua)[^)]{0,80}\)", " ", t)
    return t


def clean_idn(t: str) -> str:
    t = re.sub(r"^\s*[^-\n]{0,40}?idn times\s*-\s*", " ", t)                # dateline "balikpapan, idn times - "
    t = re.sub(r"iklan\s*-\s*scroll untuk melanjutkan", " ", t)               # widget iklan
    t = re.sub(r"idn times\s*/\s*\S+(?:\s+\S+)?", " ", t)                      # kredit foto tanpa kurung: "idn times/debbie sutrisno"
    return t


def clean_antara(t: str) -> str:
    t = re.sub(r"^.*?\(antara\)\s*-\s*", " ", t, count=1, flags=re.DOTALL)  # keterangan foto + dateline "jakarta (antara) -"
    t = re.sub(r"pewarta\s*:.*$", " ", t, flags=re.DOTALL)                    # footer: pewarta / editor / copyright
    t = re.sub(r"baca juga\s*:[^\n]*", " ", t)                                # "baca juga: <judul>" selalu 1 baris
    return t


# --- Kompas: widget "Baca juga" ditulis Title Case & menyatu dengan paragraf (tanpa pemisah baris).
# Satu-satunya petunjuk batas judulnya adalah huruf kapital, jadi dihapus SEBELUM case folding.
KONEKTOR = ("dan|di|ke|dari|untuk|yang|dengan|pada|atas|usai|jadi|jelang|soal|tak|tapi|bagi|oleh|akan|saat|kala|"
            "sebagai|atau|hingga|serta|dalam|tanpa|jika|bila|bukan|hanya|ini|itu|kini|baru|sudah|belum|masih|bisa|"
            "tidak|lebih|karena|tentang|antara|sampai|demi|lewat|via|vs|buat|ada|pun|lagi|kembali|tetap|siap|ingin|mau")
_TOK = rf"(?:[A-Z0-9][\w.,'’/\-]*[?!:;]?|(?:{KONEKTOR})|[\"“”()])"     # kata berawal kapital/angka, atau kata sambung
_BACA_JUGA = re.compile(rf"Baca juga\s*:\s*((?:{_TOK}\s+)+)")
_PROMO_KOMPASID = re.compile(rf"(?:{_TOK}\s+)+Artikel\s+Kompas\.id")        # judul promo sebelum "Artikel Kompas.id"

def hapus_widget_kompas(teks: str) -> str:
    def _ganti(m):
        toks = m.group(1).split()
        for i in range(len(toks) - 1):                       # judul diakhiri penanda "Artikel Kompas.id"
            if toks[i] == "Artikel" and toks[i + 1].lower().startswith("kompas.id"):
                return " " + " ".join(toks[i + 2:]) + " "
        for i in range(len(toks) - 1):                       # judul bertemu keterangan foto "Lihat Foto"
            if toks[i] == "Lihat" and toks[i + 1] == "Foto":
                return " " + " ".join(toks[i:]) + " "
        sisa = m.string[m.end():]
        if len(toks) > 1 and sisa[:1].islower():             # kata kapital terakhir = awal kalimat berikutnya -> kembalikan
            return " " + toks[-1] + " "
        return " "
    teks = _BACA_JUGA.sub(_ganti, teks)
    return _PROMO_KOMPASID.sub(" ", teks)


def clean_kompas(t: str) -> str:
    """Bagian Kompas yang aman dikerjakan setelah huruf kecil semua."""
    t = re.sub(r"^\s*kompas\.com\s*[-–]\s*", " ", t)                          # dateline "kompas.com - "
    t = re.sub(r"lihat foto.*?\([^)]{2,60}\)", " ", t, flags=re.DOTALL)        # keterangan foto sampai kurung kredit pertama
    t = re.sub(r"artikel kompas\.id", " ", t)                                   # sisa penanda (jika ada)
    return t


CLEANERS = {"IDN Times": clean_idn, "ANTARA": clean_antara, "Kompas": clean_kompas}

def clean_noise(text_raw: str, media: str) -> str:
    """Input: teks MENTAH. Urutan: widget Kompas (butuh huruf kapital) -> case folding -> aturan per media -> aturan umum."""
    t = hapus_widget_kompas(text_raw) if media == "Kompas" else text_raw
    t = case_folding(t)
    t = CLEANERS[media](t)
    t = clean_umum(t)
    return re.sub(r"\s+", " ", t).strip()          # rapikan spasi & newline di akhir

df["step1b_cleaned"] = [clean_noise(t, m) for t, m in zip(df["text"], df["media"])]

audit = pd.DataFrame({
    "doc_id": df["doc_id"],
    "media": df["media"],
    "panjang_sebelum": df["text"].str.len(),
    "panjang_sesudah": df["step1b_cleaned"].str.len(),
})
audit["berkurang (%)"] = ((audit["panjang_sebelum"] - audit["panjang_sesudah"]) / audit["panjang_sebelum"] * 100).round(1)

print("Rata-rata pengurangan panjang teks per media:")
display(audit.groupby("media")["berkurang (%)"].agg(["mean", "min", "max"]).round(1))
audit.head(10)

Rata-rata pengurangan panjang teks per media:


,mean,min,max
media,,,
ANTARA,20.0,12.5,32.0
IDN Times,4.4,1.0,8.2
Kompas,18.5,0.0,25.0


,doc_id,media,panjang_sebelum,panjang_sesudah,berkurang (%)
0,IDN001,IDN Times,3094,3041,1.7
1,IDN002,IDN Times,3360,3135,6.7
2,IDN003,IDN Times,2339,2264,3.2
3,IDN004,IDN Times,2583,2432,5.8
4,IDN005,IDN Times,2858,2691,5.8
5,IDN006,IDN Times,3099,3068,1.0
6,IDN007,IDN Times,1760,1730,1.7
7,IDN008,IDN Times,2202,2039,7.4
8,IDN009,IDN Times,2345,2255,3.8
9,IDN010,IDN Times,2095,2019,3.6


In [9]:
# Contoh SEBELUM vs SESUDAH untuk 1 dokumen tiap media (200 karakter pertama + terakhir)
for media in ["IDN Times", "ANTARA", "Kompas"]:
    i = df.index[df["media"] == media][0]
    print(f"=== {media} ({df.loc[i, 'doc_id']})")
    print("SEBELUM:", df.loc[i, "text"][:200].replace("\n", " "), "...")
    print("SESUDAH:", df.loc[i, "step1b_cleaned"][:200], "...")
    print()

=== IDN Times (IDN001)
SEBELUM: Balikpapan, IDN Times - Tim Adhyaksa FC yang sudah bertransformasi jadi Isenmulang Kalteng FC kembali menambah amunisi asing untuk menghadapi kompetisi Super League 2026/2027. Klub asal Kalimantan Ten ...
SESUDAH: tim adhyaksa fc yang sudah bertransformasi jadi isenmulang kalteng fc kembali menambah amunisi asing untuk menghadapi kompetisi super league 2026/2027. klub asal kalimantan tengah itu resmi mendatangk ...

=== ANTARA (ANT001)
SEBELUM: Advisor I.League Budiman Dalimunte memberikan keterangan kepada awak media di Jakarta, Senin (24/8/2026). ANTARA/Aloysius Lewokeda Jakarta (ANTARA) - Operator liga PT Liga Utama Indonesia (LIU) memast ...
SESUDAH: operator liga pt liga utama indonesia (liu) memastikan kompetisi championship akan bergulir pada 11 september, dibuka dengan pertandingan antara psis semarang melawan psps pekanbaru. "di rups (rapat u ...

=== Kompas (KMP001)
SEBELUM: KOMPAS.com - Direktur Utama I.League, Ferry Paulus, mengungkapkan piha

In [10]:
# Uji: pola noise apa saja yang MASIH lolos? (jumlah dokumen yang masih memuat pola, per media)
pola_dicurigai = ["baca juga", "lihat foto", "artikel kompas", "iklan", "pewarta", "copyright",
                  "idn times", "(antara)", ".jpg", ".jpeg", "dok.", "editor:"]
hasil = {p: df[df["step1b_cleaned"].str.contains(re.escape(p))].groupby("media").size() for p in pola_dicurigai}
sisa = pd.DataFrame(hasil).fillna(0).astype(int).T
sisa[sisa.sum(axis=1) > 0] if (sisa.sum(axis=1) > 0).any() else print("Semua pola yang diuji sudah bersih.")

Semua pola yang diuji sudah bersih.


## Tahap 2 — Tokenization

Token = rangkaian huruf/angka (`[a-z0-9]+`), sehingga tanda baca otomatis jadi pemisah.

Konsekuensinya: skor **"1-1"** atau tanggal **"23/5/2026"** terpecah menjadi token angka terpisah. Ini bukan bug, tapi akibat keputusan tokenizer.

In [11]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text)

df["step2_tokens"] = df["step1b_cleaned"].apply(tokenize)

print("SESUDAH TOKENIZATION (30 token pertama, dokumen pertama):")
print(df.loc[0, "step2_tokens"][:30])
print(f"\nTotal token dokumen ini: {len(df.loc[0, 'step2_tokens'])}")

SESUDAH TOKENIZATION (30 token pertama, dokumen pertama):
['tim', 'adhyaksa', 'fc', 'yang', 'sudah', 'bertransformasi', 'jadi', 'isenmulang', 'kalteng', 'fc', 'kembali', 'menambah', 'amunisi', 'asing', 'untuk', 'menghadapi', 'kompetisi', 'super', 'league', '2026', '2027', 'klub', 'asal', 'kalimantan', 'tengah', 'itu', 'resmi', 'mendatangkan', 'pesepak', 'bola']

Total token dokumen ini: 426


In [12]:
# Bukti dampak tokenizer: cari dokumen yang menyebut skor "x-y" pada teks mentah
pola_skor = re.compile(r"\b\d{1,2}-\d{1,2}\b")
ditemukan = 0
for _, row in df.iterrows():
    skor = pola_skor.findall(row["text"])
    if skor:
        print(f"{row['doc_id']} ({row['media']}) memuat: {sorted(set(skor))[:5]}  ->  di-token jadi angka terpisah")
        ditemukan += 1
    if ditemukan == 5:
        break

IDN002 (IDN Times) memuat: ['03-05']  ->  di-token jadi angka terpisah
IDN004 (IDN Times) memuat: ['09-11', '1-1']  ->  di-token jadi angka terpisah
IDN007 (IDN Times) memuat: ['7-9']  ->  di-token jadi angka terpisah
IDN008 (IDN Times) memuat: ['1-0', '2-1']  ->  di-token jadi angka terpisah
IDN012 (IDN Times) memuat: ['06-12']  ->  di-token jadi angka terpisah


## Tahap 3 — Filtering

Percobaan pertama: buang token **angka murni** dan token **< 3 huruf**. Cek dulu dampaknya ke term penting korpus ini, yaitu singkatan klub **"fc"**.

In [13]:
def filtering_v1(tokens):
    return [t for t in tokens if not t.isdigit() and len(t) >= 3]

hasil_v1 = df["step2_tokens"].apply(filtering_v1)

fc_sebelum = sum(t == "fc" for toks in df["step2_tokens"] for t in toks)
fc_v1 = sum(t == "fc" for toks in hasil_v1 for t in toks)
print(f"Kemunculan 'fc' sebelum filtering : {fc_sebelum}")
print(f"Kemunculan 'fc' sesudah filtering_v1 (min. 3 huruf): {fc_v1}")

Kemunculan 'fc' sebelum filtering : 83
Kemunculan 'fc' sesudah filtering_v1 (min. 3 huruf): 0


Token `fc` hilang total pada versi pertama, padahal penting untuk membedakan nama klub (Borneo **FC**, Dewa United **FC**). **Revisi**: buang angka murni saja, plus token mirip kode hex (sisa nama file gambar seperti `1842cd4ea269`).

In [14]:
HEX_LIKE = re.compile(r"(?=.*\d)[a-f0-9]{8,}")     # sisa nama file/ID acak, mis. "1842cd4ea269"

def filtering(tokens):
    # buang angka murni + token mirip kode hex (sisa UUID nama file gambar yang lolos dari pembersihan noise)
    return [t for t in tokens if not t.isdigit() and not HEX_LIKE.fullmatch(t)]

df["step3_filtered"] = df["step2_tokens"].apply(filtering)

before, after = len(df.loc[0, "step2_tokens"]), len(df.loc[0, "step3_filtered"])
print(f"Dokumen pertama: {before} token -> {after} token (berkurang {before - after})")
print("Contoh token yang dibuang:", [t for t in df.loc[0, "step2_tokens"] if t.isdigit()][:15])
print()
for term in ["liga", "persib", "fc"]:
    n = sum(t == term for toks in df["step3_filtered"] for t in toks)
    print(f"'{term}' muncul {n} kali di seluruh korpus setelah filtering")

Dokumen pertama: 426 token -> 395 token (berkurang 31)
Contoh token yang dibuang: ['2026', '2027', '22', '1996', '177', '30', '1', '2015', '20', '2016', '2018', '26', '2018', '2021', '2024']

'liga' muncul 83 kali di seluruh korpus setelah filtering
'persib' muncul 183 kali di seluruh korpus setelah filtering
'fc' muncul 83 kali di seluruh korpus setelah filtering


## Tahap 4 — Stopword Removal

Memakai stopword bawaan Sastrawi. Cek dulu apakah kata negasi ikut terbuang, lalu terapkan daftar final (dikurangi negasi, ditambah stopword boilerplate dari Bagian 1).

In [15]:
default_stopwords = set(StopWordRemoverFactory().get_stop_words())
print(f"Jumlah stopword default Sastrawi: {len(default_stopwords)}")
for w in ["tidak", "bukan", "belum", "tanpa"]:
    print(f"  '{w}' termasuk stopword default? -> {w in default_stopwords}")

stopwords_final = set(default_stopwords)
if PERTAHANKAN_NEGASI:
    stopwords_final -= KATA_NEGASI
if PAKAI_STOPWORD_TAMBAHAN:
    stopwords_final |= STOPWORD_TAMBAHAN
print(f"\nJumlah stopword final yang dipakai: {len(stopwords_final)}")

Jumlah stopword default Sastrawi: 809
  'tidak' termasuk stopword default? -> True
  'bukan' termasuk stopword default? -> True
  'belum' termasuk stopword default? -> True
  'tanpa' termasuk stopword default? -> True

Jumlah stopword final yang dipakai: 827


In [16]:
def stopword_removal(tokens, stopwords):
    return [t for t in tokens if t not in stopwords]

df["step4_no_stopwords"] = df["step3_filtered"].apply(lambda toks: stopword_removal(toks, stopwords_final))

removed = [t for t in df.loc[0, "step3_filtered"] if t not in df.loc[0, "step4_no_stopwords"]]
print(f"Dokumen pertama: {len(df.loc[0, 'step3_filtered'])} token -> {len(df.loc[0, 'step4_no_stopwords'])} token")
print("Contoh stopword yang dibuang:", sorted(set(removed))[:20])

Dokumen pertama: 395 token -> 268 token
Contoh stopword yang dibuang: ['akhir', 'akhirnya', 'antara', 'asal', 'awal', 'bagi', 'baru', 'beberapa', 'bersama', 'com', 'dalam', 'dan', 'dapat', 'dari', 'dengan', 'di', 'dimulai', 'diri', 'dua', 'februari']


## Tahap 5 — Stemming

Sastrawi Stemmer (Nazief & Adriani). Stemming kata yang sama berulang itu mubazir, jadi hasilnya di-*cache* dengan `lru_cache`.

In [17]:
stemmer = StemmerFactory().create_stemmer()

@lru_cache(maxsize=None)
def stem_cached(word: str) -> str:
    return stemmer.stem(word)

def stemming(tokens):
    return [stem_cached(t) for t in tokens]

df["step5_stemmed"] = df["step4_no_stopwords"].apply(stemming)

print("SEBELUM:", df.loc[0, "step4_no_stopwords"][:15])
print("SESUDAH:", df.loc[0, "step5_stemmed"][:15])

SEBELUM: ['tim', 'adhyaksa', 'fc', 'bertransformasi', 'isenmulang', 'kalteng', 'fc', 'menambah', 'amunisi', 'asing', 'menghadapi', 'kompetisi', 'super', 'league', 'klub']
SESUDAH: ['tim', 'adhyaksa', 'fc', 'transformasi', 'isenmulang', 'kalteng', 'fc', 'tambah', 'amunisi', 'asing', 'hadap', 'kompetisi', 'super', 'league', 'klub']


In [18]:
# Cek kualitas stemming di SELURUH korpus
pasangan = Counter()
for sebelum, sesudah in zip(df["step4_no_stopwords"], df["step5_stemmed"]):
    for a, b in zip(sebelum, sesudah):
        if a != b:
            pasangan[(a, b)] += 1
print(f"Jumlah pasangan unik (kata asli -> hasil stem) yang berubah: {len(pasangan)}\n")

# Kandidat OVER-STEMMING: hasil stem sangat pendek (<= 3 huruf) padahal kata aslinya >= 5 huruf
over = sorted([(a, b, n) for (a, b), n in pasangan.items() if len(b) <= 3 and len(a) >= 5],
              key=lambda x: -x[2])
print("Kandidat over-stemming (kata asli -> stem, frekuensi):")
for a, b, n in over[:15]:
    print(f"  {a!r} -> {b!r}  ({n}x)")

# Cek manual: nama klub/tempat/orang yang rentan terpotong
print("\nCek nama diri yang rentan:")
for w in ["bali", "milan", "toni", "borneo", "persebaya", "arema", "bhayangkara", "malut", "kalteng", "lampung"]:
    print(f"  {w!r} -> {stem_cached(w)!r}")

Jumlah pasangan unik (kata asli -> hasil stem) yang berubah: 1101

Kandidat over-stemming (kata asli -> stem, frekuensi):
  'mengaku' -> 'aku'  (18x)
  'timnya' -> 'tim'  (6x)
  'terkesan' -> 'kes'  (4x)
  'mengajukan' -> 'aju'  (3x)
  'diajukan' -> 'aju'  (3x)
  'berisi' -> 'isi'  (3x)
  'setim' -> 'tim'  (3x)
  'keberadaan' -> 'ada'  (3x)
  'kemauan' -> 'mau'  (3x)
  'pengajuan' -> 'aju'  (2x)
  'memesan' -> 'mes'  (2x)
  'diisukan' -> 'isu'  (2x)
  'berbau' -> 'bau'  (2x)
  'kerim' -> 'rim'  (2x)
  'denis' -> 'den'  (2x)

Cek nama diri yang rentan:
  'bali' -> 'bal'
  'milan' -> 'mil'
  'toni' -> 'ton'
  'borneo' -> 'borneo'
  'persebaya' -> 'baya'
  'arema' -> 'arema'
  'bhayangkara' -> 'bhayangkara'
  'malut' -> 'malut'
  'kalteng' -> 'kalteng'
  'lampung' -> 'lampung'


In [19]:
# Kandidat UNDER-STEMMING: ejaan berbeda untuk kata yang sama
print("Varian ejaan yang tidak disatukan:")
for w1, w2 in [("jersey", "jersei"), ("internasional", "international"), ("regular", "reguler"), ("striker", "penyerang")]:
    print(f"  {w1!r} -> {stem_cached(w1)!r}   |   {w2!r} -> {stem_cached(w2)!r}")

Varian ejaan yang tidak disatukan:
  'jersey' -> 'jersey'   |   'jersei' -> 'jersei'
  'internasional' -> 'internasional'   |   'international' -> 'international'
  'regular' -> 'regular'   |   'reguler' -> 'reguler'
  'striker' -> 'striker'   |   'penyerang' -> 'serang'


## Pipeline Lengkap

Semua tahap digabung menjadi satu fungsi, dijalankan ke seluruh 60 dokumen. Fungsi ini butuh `media` karena aturan noise berbeda per media.

In [20]:
def preprocess_pipeline(raw_text: str, media: str) -> list[str]:
    text = clean_noise(raw_text, media)          # sudah mencakup case folding (lihat Tahap 1b)
    tokens = tokenize(text)
    tokens = filtering(tokens)
    tokens = stopword_removal(tokens, stopwords_final)
    tokens = stemming(tokens)
    return tokens

df["tokens_final"] = [preprocess_pipeline(t, m) for t, m in zip(df["text"], df["media"])]
df["text_final"] = df["tokens_final"].apply(" ".join)

# Konsistensi: hasil pipeline harus sama dengan hasil tahap demi tahap
assert df["tokens_final"].equals(df["step5_stemmed"]), "Pipeline lengkap berbeda dari hasil tahap-per-tahap!"
print("OK: pipeline lengkap identik dengan hasil tahap demi tahap.")

df[["doc_id", "media", "title", "tokens_final"]].head(10)

OK: pipeline lengkap identik dengan hasil tahap demi tahap.


,doc_id,media,title,tokens_final
0,IDN001,IDN Times,"Dari Boca Juniors ke Kalteng, Adriel Fernandez Bomber Baru Isenmulang","[tim, adhyaksa, fc, transformasi, isenmulang, kalteng, fc, tambah, amunisi, asing, hadap, kompetisi, super, league, ..."
1,IDN002,IDN Times,Pesona Gelandang Gambia di Asia Bikin Isenmulang Kalteng FC Kepincut,"[isenmulang, kalteng, fc, tambah, kuat, skuadnya, hadap, kompetisi, super, league, klub, kalimantan, tengah, resmi, ..."
2,IDN003,IDN Times,Persija dan Se'Indonesia Perpanjang Kontrak Hingga 2 Musim,"[persija, jakarta, indonesia, lanjut, musim, indonesia, mejeng, jersey, persija, musim, resmi, lanjut, persija, indo..."
3,IDN004,IDN Times,"I.League: Persija Minta Jamu Persib Lebih Dulu, Main di Jakarta","[league, konfirmasi, aju, persija, jakarta, jamu, persib, bandung, super, league, musim, duel, upaya, jakarta, dasar..."
4,IDN005,IDN Times,Jadwal Super League Musim 2026/27 Adopsi Kompetisi Top Dunia,"[league, adopsi, sistem, jadwal, modern, sepak, bola, inggris, jepang, bijak, ambil, susun, kalender, tanding, struk..."
5,IDN006,IDN Times,Cari Talenta Baru Lewat Batam Prime International GFF 2026,"[turnamen, sepak, bola, usia, nama, indonesia, batam, prime, international, grassroot, football, festival, ajang, st..."
6,IDN007,IDN Times,Malut United Resmi Berganti Nama Jadi Java United FC,"[musim, momen, malut, united, bawa, nama, malu, utara, ganti, identitas, klub, serta, markas, jelang, arung, super, ..."
7,IDN008,IDN Times,"Semifinal Piala Presiden Panas, Derby Jatim Hingga Persib vs Persija","[piala, presiden, dominasi, klub, lokal, final, tim, laga, super, league, musim, persija, jakarta, tim, lolos, semif..."
8,IDN009,IDN Times,Semen Padang Minta Semua Tim Championship Tak Curang,"[presiden, semen, padang, andre, rosiade, tim, tim, mentas, championship, saing, sehat, harap, curang, mari, kompeti..."
9,IDN010,IDN Times,Ini Kesan Shin Tae Yong Usai Pimpin Latihan Perdana Persija,"[shin, tae, yong, kerja, latih, persija, jakarta, perdana, pimpin, latih, tim, bojongsari, depok, rekrut, anyar, iku..."


## Dampak Preprocessing secara Kuantitatif

In [21]:
ringkasan = pd.DataFrame({
    "doc_id": df["doc_id"],
    "media": df["media"],
    "token_mentah": df["text"].apply(lambda t: len(t.split())),
    "token_akhir": df["tokens_final"].apply(len),
})
ringkasan["pengurangan (%)"] = ((ringkasan["token_mentah"] - ringkasan["token_akhir"]) / ringkasan["token_mentah"] * 100).round(1)

print("Per media:")
display(ringkasan.groupby("media")[["token_mentah", "token_akhir", "pengurangan (%)"]].mean().round(1))

print("Pengurangan tertinggi & terendah:")
display(ringkasan.sort_values("pengurangan (%)").iloc[[0, -1]])

print(f"Vocabulary (term unik): sebelum = {len(set(w for t in df['text'] for w in t.lower().split()))}, "
      f"sesudah = {len(set(w for toks in df['tokens_final'] for w in toks))}")

Per media:


,token_mentah,token_akhir,pengurangan (%)
media,,,
ANTARA,367.1,163.8,55.3
IDN Times,374.1,210.8,43.4
Kompas,499.4,224.9,54.9


Pengurangan tertinggi & terendah:


,doc_id,media,token_mentah,token_akhir,pengurangan (%)
0,IDN001,IDN Times,420,268,36.2
36,ANT017,ANTARA,353,114,67.7


Vocabulary (term unik): sebelum = 5068, sesudah = 2060


In [22]:
# Term paling sering per media + cek sisa noise
semua = Counter(w for toks in df["tokens_final"] for w in toks)
for media in ["IDN Times", "ANTARA", "Kompas"]:
    c = Counter(w for toks in df.loc[df["media"] == media, "tokens_final"] for w in toks)
    print(f"{media:10s}:", [w for w, _ in c.most_common(15)])

kandidat_noise = ["idn", "times", "antara", "kompas", "com", "jpeg", "jpg", "dok", "pewarta", "copyright", "lihat", "baca", "artikel", "iklan"]
print("\nSisa kandidat noise di vocabulary akhir:", {w: semua[w] for w in kandidat_noise if semua[w]} or "tidak ada")

IDN Times : ['main', 'klub', 'persib', 'league', 'musim', 'tim', 'indonesia', 'persija', 'fc', 'kompetisi', 'latih', 'super', 'piala', 'presiden', 'bola']
ANTARA    : ['main', 'musim', 'tim', 'klub', 'latih', 'persib', 'kompetisi', 'indonesia', 'bandung', 'liga', 'jalan', 'milik', 'persija', 'united', 'harap']
Kompas    : ['main', 'indonesia', 'bola', 'tim', 'persib', 'league', 'sepak', 'klub', 'kompetisi', 'tidak', 'musim', 'persija', 'liga', 'latih', 'juara']

Sisa kandidat noise di vocabulary akhir: {'dok': 2, 'baca': 2, 'artikel': 3}


## Menyimpan Hasil untuk Praktikum Berikutnya

In [25]:
df_out = df[["doc_id", "media", "title", "text_final"]]
df_out.to_csv("hasil_preprocessing_project.csv", index=False)
print(f"Tersimpan: hasil_preprocessing_project.csv ({len(df_out)} dokumen)")

try:
    from google.colab import files
    files.download("hasil_preprocessing_project.csv")
except ImportError:
    pass

df_out.head()

Tersimpan: hasil_preprocessing_project.csv (60 dokumen)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,doc_id,media,title,text_final
0,IDN001,IDN Times,"Dari Boca Juniors ke Kalteng, Adriel Fernandez Bomber Baru Isenmulang",tim adhyaksa fc transformasi isenmulang kalteng fc tambah amunisi asing hadap kompetisi super league klub kalimantan...
1,IDN002,IDN Times,Pesona Gelandang Gambia di Asia Bikin Isenmulang Kalteng FC Kepincut,isenmulang kalteng fc tambah kuat skuadnya hadap kompetisi super league klub kalimantan tengah resmi gelandang gambi...
2,IDN003,IDN Times,Persija dan Se'Indonesia Perpanjang Kontrak Hingga 2 Musim,persija jakarta indonesia lanjut musim indonesia mejeng jersey persija musim resmi lanjut persija indonesia persija ...
3,IDN004,IDN Times,"I.League: Persija Minta Jamu Persib Lebih Dulu, Main di Jakarta",league konfirmasi aju persija jakarta jamu persib bandung super league musim duel upaya jakarta dasar draf jadwal re...
4,IDN005,IDN Times,Jadwal Super League Musim 2026/27 Adopsi Kompetisi Top Dunia,league adopsi sistem jadwal modern sepak bola inggris jepang bijak ambil susun kalender tanding struktur presiden di...
